<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/v3/llm-inference/speculative-decoding/speculative-decoding.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("speculative-decoding", ...)`


# Question 18: Implement Speculative Decoding

### Difficulty: Hard

### Problem Statement

Autoregressive decoding is slow because each token requires a full forward pass through the model. **Speculative decoding** speeds this up by using a small, fast **draft model** to propose K tokens, then verifying all K tokens in a **single** forward pass of the larger **target model**.

The key insight is that the target model can score all K proposed tokens in parallel (since we know the draft tokens), and we use **rejection sampling** to decide which draft tokens to accept. This guarantees that the output distribution matches exactly what the target model would produce with standard autoregressive decoding.

Your task is to:
1. Create a small `DraftModel` and a larger `TargetModel`.
2. Implement the speculative decoding algorithm with rejection sampling.
3. Compare speculative decoding to standard (target-only) decoding.

---

### Requirements

1. **`DraftModel`**: A small language model (e.g., 1 hidden layer with a small hidden dim) that maps token ids `(batch_size, seq_len)` to next-token logits `(batch_size, seq_len, vocab_size)`.

2. **`TargetModel`**: A larger language model (e.g., 3 hidden layers with a larger hidden dim), same interface, that produces the "true" distribution.

   The logits at position `t` must depend only on tokens `0..t`, because verification reads several positions out of ONE target pass. Per-position MLPs (effectively bigram models: position `t` sees only token `t`) are enough to exercise the algorithm; with real transformers both models would also keep KV caches and roll back the rejected positions.

3. **`speculative_decode(draft_model, target_model, input_ids, K=5, ...)`**:
   - Use the draft model to autoregressively sample K candidate tokens `x_1..x_K`, keeping the draft's full next-token distribution `q_i` at each step.
   - Run the target model on the full sequence (input + K draft tokens) in one forward pass. Position `prompt_len - 1 + i` gives the target distribution `p_i` that scores draft token `x_i` (0-indexed), and position `prompt_len + K - 1` gives the distribution for the token after the last draft.
   - For each draft token in order, accept with probability `min(1, p_i(x_i) / q_i(x_i))`. On the **first** rejection, sample a replacement from `normalize(max(0, p_i - q_i))` and stop.
   - If all K drafts are accepted, sample one **bonus** token from the target's last position.
   - Return `(tokens, num_accepted)`: the accepted drafts followed by exactly one more token (the replacement or the bonus) — between 1 and K+1 tokens per target forward pass.

4. **Validation**: Show the acceptance rate across multiple runs. Verify empirically that the output follows the target model's distribution. Compare wall-clock time of speculative vs standard decoding.

---

### Constraints

- Use only PyTorch operations.
- The rejection sampling must be mathematically correct: on rejection at position i, sample from `max(0, p_target - p_draft)` normalized.
- Both models share the same vocabulary size.

---

### Why the output distribution is exactly the target's

At one position, with target distribution $p$ and draft distribution $q$, token $x$ is emitted either by being drafted and accepted, or by a rejection followed by a draw from the residual $r(x) = \frac{\max(0,\, p(x) - q(x))}{\sum_y \max(0,\, p(y) - q(y))}$:

$$P(x) = q(x)\min\!\left(1, \tfrac{p(x)}{q(x)}\right) + P(\text{reject})\, r(x) = \min(p(x), q(x)) + \max(0,\, p(x) - q(x)) = p(x),$$

because $P(\text{reject}) = 1 - \sum_y \min(p(y), q(y)) = \sum_y \max(0,\, p(y) - q(y))$ cancels the residual's normaliser. Every later position is scored on a prefix that was accepted, so by induction the whole sequence follows the target model; the bonus token is simply an exact sample from $p$ that the verification pass already paid for (Leviathan et al., 2023; Chen et al., 2023).

- **Expected speedup**: with per-token acceptance rate $\alpha = \sum_x \min(p(x), q(x))$, a round emits $\frac{1 - \alpha^{K+1}}{1 - \alpha}$ tokens on average for one target pass. It pays off in wall-clock time only when scoring K+1 positions costs about as much as scoring one (memory-bandwidth-bound decoding on a GPU) and the draft is much cheaper than the target — on CPU with toy models the timings below are only illustrative.
- **Greedy variant** (temperature → 0): accept a draft token iff it equals the target's argmax; on the first mismatch emit the target's argmax. The output is then token-for-token identical to greedy decoding with the target.

---

### Company Tags

Google, DeepMind, Anthropic, Apple

---

<details>
  <summary>Hint</summary>

  - The draft model generates K tokens one at a time, storing both the tokens and the draft probabilities at each step.
  - The target model scores all K+1 positions in a single forward pass (input + K draft tokens). The +1 is for the token after the last draft token.
  - For rejection sampling: draw `r ~ Uniform(0, 1)`. Accept draft token `x` if `r < p_target(x) / p_draft(x)`. Otherwise, sample from `normalize(max(0, p_target - p_draft))` — here `p_target` and `p_draft` are the full distributions over the vocabulary at that position, not just the drafted token's probability.
  - If all K tokens are accepted, you get a bonus: sample one more token from the target model's distribution at the last position (index `prompt_len + K - 1`), which the verification pass already computed.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

In [ ]:
# Configuration
torch.manual_seed(42)

vocab_size = 128
d_model_draft = 64
d_model_target = 256
max_seq_len = 64
K = 5  # Number of speculative tokens

device = "cpu"
print(f"Vocab size: {vocab_size}, Draft dim: {d_model_draft}, Target dim: {d_model_target}, K: {K}")

In [ ]:
class DraftModel(nn.Module):
    """
    Small, fast model that proposes candidate tokens.
    Simple architecture: Embedding -> Linear -> ReLU -> Linear -> logits
    """
    
    def __init__(self, vocab_size, d_model):
        super().__init__()
        # TODO: Define layers
        # - Embedding(vocab_size, d_model)
        # - Linear(d_model, d_model)
        # - Linear(d_model, vocab_size)  (output logits)
        ...
    
    def forward(self, input_ids):
        """
        Args:
            input_ids: (batch_size, seq_len) token indices
        Returns:
            logits: (batch_size, seq_len, vocab_size)
        """
        # TODO: Forward pass
        ...


class TargetModel(nn.Module):
    """
    Larger, slower model that produces the "true" distribution.
    Architecture: Embedding -> 3x(Linear -> ReLU) -> Linear -> logits
    """
    
    def __init__(self, vocab_size, d_model):
        super().__init__()
        # TODO: Define layers (larger/deeper than DraftModel)
        ...
    
    def forward(self, input_ids):
        """
        Args:
            input_ids: (batch_size, seq_len) token indices
        Returns:
            logits: (batch_size, seq_len, vocab_size)
        """
        # TODO: Forward pass
        ...


print("Model classes defined. Fill in the TODO sections above.")

In [ ]:
def standard_decode(model, input_ids, num_tokens, temperature=1.0):
    """
    Standard autoregressive decoding from the target model.
    Generates one token at a time.
    
    Args:
        model: The target model
        input_ids: (1, prompt_len) initial token indices
        num_tokens: Number of tokens to generate
        temperature: Sampling temperature
        
    Returns:
        generated_ids: (1, prompt_len + num_tokens) full sequence
    """
    # TODO: Implement standard autoregressive decoding
    # For each step:
    #   1. Forward pass through model
    #   2. Get logits for last position
    #   3. Apply temperature and softmax to get probabilities
    #   4. Sample next token
    #   5. Append to sequence
    ...


def speculative_decode(draft_model, target_model, input_ids, K=5, temperature=1.0):
    """
    Speculative decoding: draft model proposes K tokens, target model verifies.
    
    Args:
        draft_model: Small, fast model for proposing tokens
        target_model: Large, accurate model for verification
        input_ids: (1, prompt_len) initial token indices
        K: Number of speculative tokens to propose
        temperature: Sampling temperature
        
    Returns:
        output_tokens: List of emitted token ids: the accepted draft tokens followed by
                       exactly one more token (the residual resample after the first
                       rejection, or the bonus token if all K drafts were accepted)
        num_accepted: Number of draft tokens accepted (out of K)
    """
    # TODO: Implement speculative decoding
    #
    # Step 1: Draft phase
    #   - Use draft_model to autoregressively generate K tokens
    #   - Store each draft token and the draft's FULL next-token distribution q
    #
    # Step 2: Verification phase
    #   - Run target_model on the full sequence (prompt + K draft tokens) in ONE forward pass
    #   - Draft token i is scored by the target distribution at position prompt_len - 1 + i
    #
    # Step 3: Rejection sampling
    #   For each draft token i (in order):
    #     - p_draft = draft probability of chosen token
    #     - p_target = target probability of chosen token
    #     - Draw r ~ Uniform(0, 1)
    #     - If r < min(1, p_target / p_draft): ACCEPT
    #     - Else: REJECT
    #       - Sample replacement from: normalize(max(0, p - q)), using the full
    #         target / draft distributions at this position
    #       - Return accepted tokens so far + replacement token
    #
    # Step 4: If all K tokens accepted, sample one bonus token from the target's
    #         distribution at the last position (prompt_len + K - 1)
    #
    ...


print("Decoding functions defined. Fill in the TODO sections above.")

In [ ]:
# ============================================================
# Validation & Benchmarking
# ============================================================

torch.manual_seed(42)
draft_model = DraftModel(vocab_size, d_model_draft).to(device).eval()
target_model = TargetModel(vocab_size, d_model_target).to(device).eval()

print(f"Draft model params:  {sum(p.numel() for p in draft_model.parameters()):,}")
print(f"Target model params: {sum(p.numel() for p in target_model.parameters()):,}")
print()

# Generate tokens using speculative decoding (multiple rounds)
prompt = torch.randint(0, vocab_size, (1, 5), device=device)
total_generated = 0
total_accepted = 0
num_rounds = 20

current_ids = prompt.clone()
with torch.no_grad():
    for round_idx in range(num_rounds):
        accepted_tokens, num_accepted = speculative_decode(
            draft_model, target_model, current_ids, K=K, temperature=1.0
        )
        total_generated += len(accepted_tokens)
        total_accepted += num_accepted
        # Append accepted tokens to the sequence
        new_tokens = torch.tensor([accepted_tokens], device=device)
        current_ids = torch.cat([current_ids, new_tokens], dim=1)

acceptance_rate = total_accepted / (num_rounds * K)
print(f"Total tokens generated: {total_generated}")
print(f"Draft tokens accepted: {total_accepted} / {num_rounds * K}")
print(f"Acceptance rate: {acceptance_rate:.1%}")
print(f"Avg tokens per round: {total_generated / num_rounds:.1f} (max possible: {K + 1})")
print()

# --- Benchmark: speculative vs standard ---
num_tokens_to_gen = 50

# Standard decoding
prompt = torch.randint(0, vocab_size, (1, 5), device=device)
start = time.time()
with torch.no_grad():
    std_output = standard_decode(target_model, prompt, num_tokens_to_gen)
std_time = time.time() - start

# Speculative decoding
current_ids = prompt.clone()
spec_count = 0
start = time.time()
with torch.no_grad():
    while spec_count < num_tokens_to_gen:
        accepted_tokens, _ = speculative_decode(draft_model, target_model, current_ids, K=K)
        remaining = num_tokens_to_gen - spec_count
        tokens_to_add = accepted_tokens[:remaining]
        new_tokens = torch.tensor([tokens_to_add], device=device)
        current_ids = torch.cat([current_ids, new_tokens], dim=1)
        spec_count += len(tokens_to_add)
spec_time = time.time() - start

print(f"Standard decoding: {std_time:.4f}s for {num_tokens_to_gen} tokens")
print(f"Speculative decoding: {spec_time:.4f}s for {num_tokens_to_gen} tokens")
print(f"Speedup: {std_time / spec_time:.2f}x")

In [ ]:
# ============================================================
# Correctness: speculative decoding must not change the output distribution
# ============================================================
# (a) First emitted token, on the models above: its distribution must be the TARGET's
#     next-token distribution, not the draft's. A low temperature sharpens both models
#     so that they clearly disagree. For scale, we also measure how far 1000 samples drawn
#     directly from the target land from it (pure sampling noise).
torch.manual_seed(0)
temp = 0.3
prompt = torch.randint(0, vocab_size, (1, 5), device=device)
with torch.no_grad():
    p_next = F.softmax(target_model(prompt)[0, -1] / temp, dim=-1)
    q_next = F.softmax(draft_model(prompt)[0, -1] / temp, dim=-1)
    n_rounds = 1000
    counts = torch.zeros(vocab_size)
    for _ in range(n_rounds):
        tokens, _ = speculative_decode(draft_model, target_model, prompt, K=2, temperature=temp)
        counts[tokens[0]] += 1
first_token_freq = counts / n_rounds
tv = lambda a, b: 0.5 * (a - b).abs().sum().item()  # total-variation distance
noise_freq = torch.multinomial(p_next, n_rounds, replacement=True).bincount(minlength=vocab_size) / n_rounds
print(f"(a) TV(draft, target) = {tv(q_next, p_next):.3f}; "
      f"TV(speculative, target) = {tv(first_token_freq, p_next):.3f}; "
      f"TV(speculative, draft) = {tv(first_token_freq, q_next):.3f}; "
      f"noise level TV(direct target samples, target) = {tv(noise_freq, p_next):.3f}")
assert tv(first_token_freq, p_next) < tv(first_token_freq, q_next), "the first emitted token follows the DRAFT, not the target"

# (b) The whole guarantee, exactly: on hand-built bigram models over a 3-token vocabulary,
#     the joint distribution of the first 3 generated tokens (K=2, so the bonus token is
#     included) must equal the target's, which we can enumerate.
import itertools
P = torch.tensor([[0.6, 0.3, 0.1], [0.2, 0.2, 0.6], [0.1, 0.7, 0.2]])   # target: P[prev, next]
Q = torch.tensor([[0.2, 0.3, 0.5], [0.5, 0.4, 0.1], [0.4, 0.2, 0.4]])   # draft

class BigramLM(nn.Module):
    def __init__(self, table):
        super().__init__()
        self.register_buffer("log_table", table.log())
    def forward(self, input_ids):
        return self.log_table[input_ids]          # (B, S, 3)

seqs = list(itertools.product(range(3), repeat=3))
exact = torch.tensor([P[0, a] * P[a, b] * P[b, c] for a, b, c in seqs])   # prompt ends in token 0
counts = torch.zeros(len(seqs))
n_runs = 5000
with torch.no_grad():
    for _ in range(n_runs):
        ids, out = torch.tensor([[1, 0]]), []
        while len(out) < 3:
            tokens, _ = speculative_decode(BigramLM(Q), BigramLM(P), ids, K=2)
            out += tokens
            ids = torch.cat([ids, torch.tensor([tokens])], dim=1)
        counts[seqs.index(tuple(out[:3]))] += 1
print(f"(b) TV(speculative joint over 3 tokens, exact target joint) = {tv(counts / n_runs, exact):.3f} "
      f"(sampling noise alone: ~0.025)")
assert tv(counts / n_runs, exact) < 0.07, "speculative decoding changed the output distribution"
print("Speculative decoding samples exactly from the target model's distribution.")